# SEM grinding wheel -> Abaqus + CAD (semgrit)

Measures abrasive grains in SEM micrographs, then exports:

- an **Abaqus `.inp`** containing **every** grit on the wheel (the main deliverable), and
- **STEP CAD** of the wheel with grits on it, for checking the geometry in SOLIDWORKS.

Replaces the original `SEM_WHEEL.ipynb`, whose scale-bar detection locked onto the white
databar panel (1019 px) instead of the bar (68 px), making every measurement **14.9x too
small at 10 kX and 29.9x at 5 kX**. This version reads the exact pixel size from the Zeiss
TIFF metadata and uses the scale bar only as an independent cross-check.

Run the cells in order.


## 1. Install dependencies


In [ ]:
!pip install -q shapely mapbox-earcut
print('dependencies ready')


## 2. Upload the code bundle

Upload **`semgrit_colab.zip`** (it sits next to this notebook). Shipping the package as a
zip keeps one source of truth: this is byte-identical to the version that passes all 36
verification checks.


In [ ]:
import os, sys, zipfile
from google.colab import files

if not os.path.isdir('semgrit'):
    up = files.upload()          # choose semgrit_colab.zip
    name = next(k for k in up if k.endswith('.zip'))
    with zipfile.ZipFile(name) as z:
        z.extractall('.')

sys.path.insert(0, os.getcwd())
import semgrit
print('semgrit', semgrit.__version__, 'loaded')


## 3. Upload your SEM images

Upload the **original `.tif` files with metadata intact**. Do not screenshot them or
re-save as PNG/JPEG: that strips the embedded calibration and the run will stop with an
error rather than silently guess a scale.


In [ ]:
import glob, os
from google.colab import files

os.makedirs('images', exist_ok=True)
up = files.upload()
for name in up:
    os.replace(name, os.path.join('images', name))
imgs = sorted(glob.glob('images/*.tif')) + sorted(glob.glob('images/*.tiff'))
print(f'{len(imgs)} images ready:')
for i in imgs:
    print('   ', os.path.basename(i))


## 4. Measure the grains

`edge_radius_um` is the **cutting edge radius**. A mathematically sharp edge has zero
radius, which is a **stress singularity** in FEA -- stress grows without bound as the mesh
is refined, so results never converge. It also sets the minimum chip thickness physically.
A good starting point is ~10% of the measured d50 grain size.

`max_vertices` is the simplification lever: lower means fewer faces per grain, which is
what pays for the extra geometry the rounding adds.


In [ ]:
edge_radius_um = 0.35  #@param {type:"number"}
max_vertices = 8  #@param {type:"integer"}
simplify_um = 0.25  #@param {type:"number"}

cmd = (f'python -W ignore -m semgrit analyze "images/*.tif" -o results'
       f' --edge-radius-um {edge_radius_um} --arc-segments 1'
       f' --max-vertices {max_vertices} --simplify-um {simplify_um} --verify')
print(cmd, chr(10))
!{cmd}


### Review the segmentation

**Green = interior grain** (used for statistics and CAD). **Red = touches the image
border**, so it is truncated and excluded from the size distribution.


In [ ]:
import glob, os
import matplotlib.pyplot as plt, matplotlib.image as mpimg

ovs = sorted(glob.glob('results/*_segmentation.png'))
n = min(len(ovs), 4)
if n:
    fig, axes = plt.subplots(n, 1, figsize=(13, 9 * n))
    axes = [axes] if n == 1 else list(axes)
    for ax, p in zip(axes, ovs[:n]):
        ax.imshow(mpimg.imread(p)); ax.axis('off')
        ax.set_title(os.path.basename(p).replace('_segmentation.png', ''))
    plt.tight_layout(); plt.show()
else:
    print('no overlays found - did step 4 run?')


### Size distribution


In [ ]:
import json, glob
import numpy as np, pandas as pd, matplotlib.pyplot as plt

s = json.load(open('results/summary.json'))['pooled']
print(f"images               : {s['n_images']}")
print(f"grains measured      : {s['n_grains_total']} ({s['n_grains_used']} interior)")
print(f"areal density in SEM : {s['areal_density_per_mm2']:.0f} grains/mm2")
for k in ('equivalent_diameter_um', 'feret_max_um', 'feret_min_um', 'aspect_ratio', 'solidity'):
    d = s[k]
    print(f"{k:22s}: d10={d['d10']:7.3f}  d50={d['d50']:7.3f}  d90={d['d90']:7.3f}")

df = pd.concat([pd.read_csv(f) for f in glob.glob('results/*_grains.csv')], ignore_index=True)
inter = df[~df.touches_border]
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
ax[0].hist(inter.equivalent_diameter_um, bins=30, color='#3b78c3'); ax[0].set_xlabel('equivalent diameter (um)')
ax[1].hist(inter.aspect_ratio, bins=30, color='#c3733b');           ax[1].set_xlabel('aspect ratio')
ax[2].hist(inter.min_corner_angle_deg, bins=30, color='#4b9e5f');   ax[2].set_xlabel('sharpest corner (deg)')
for a in ax:
    a.set_ylabel('count'); a.grid(alpha=.3)
plt.suptitle(f'{len(inter)} interior grains'); plt.tight_layout(); plt.show()


## 5. Choose the wheel

Defaults are a standard ISO 525 / ANSI B74.13 flat wheel: **Ø200 x 20 x 31.75 mm**, 30 deg
sector, 5 mm abrasive rim.

### About `grains_per_mm2`

This is the one number worth thinking about. Fine grit implies an enormous count: at a true
C100 diamond concentration, 3.4 um grit works out to ~27,000 grains/mm2, and a 30 deg x
20 mm surface is 1047 mm2 -- about **28 million grains**, which is not tractable.

So either accept a density below the real concentration over the full sector, or keep true
density over a smaller sector. Run the next cell to see the trade-off for *your* grit before
committing.


In [ ]:
import json, math, pickle
from semgrit.wheel import WheelSpec, GrainPopulationSpec, estimate_grain_count

lib = pickle.load(open('results/grain_library.pkl', 'rb'))
faces = sum(len(s.faces) for s in lib['solids']) / len(lib['solids'])
print(f'grain library: {len(lib["solids"])} shapes, {faces:.0f} faces each\n')
print(f'{"sector x width":>22} {"area mm2":>9} {"grains at true C100":>21}')
for sec, wid in ((30, 20), (30, 2), (5, 2), (1, 1), (0.2, 0.2)):
    sp = WheelSpec(diameter_mm=200, width_mm=wid, sector_deg=sec,
                   rim_depth_mm=min(5, wid / 2))
    n, info = estimate_grain_count(sp, GrainPopulationSpec(concentration=100.0,
                                                          max_grains=10**9), lib['solids'])
    print(f'{f"{sec} deg x {wid} mm":>22} {sp.surface_area_mm2:9.2f} {info["uncapped_grains"]:>21,}')
print(f'\ntrue C100 areal density = {info["areal_density_per_mm2"]:.0f} grains/mm2')


In [ ]:
wheel_diameter_mm = 200  #@param {type:"number"}
wheel_width_mm = 20  #@param {type:"number"}
bore_diameter_mm = 31.75  #@param {type:"number"}
sector_deg = 30  #@param {type:"number"}
rim_depth_mm = 5  #@param {type:"number"}
grains_per_mm2 = 100  #@param {type:"number"}
max_grains = 105000  #@param {type:"integer"}
grain_element = "R3D3"  #@param ["R3D3", "C3D4"]
grain_material = "diamond"  #@param ["diamond", "b4c", "cbn"]
bond_material = "vitrified_bond"  #@param ["vitrified_bond", "resin_bond", "metal_bond"]

WHEEL_ARGS = (f' --diameter {wheel_diameter_mm} --width {wheel_width_mm}'
              f' --sector {sector_deg} --rim-depth {rim_depth_mm}'
              f' --hub-diameter {bore_diameter_mm}'
              f' --areal-density {grains_per_mm2} --max-grains {max_grains}'
              f' --grain-element {grain_element}'
              f' --grain-material {grain_material} --bond-material {bond_material}')
print('wheel settings:', WHEEL_ARGS)


## 6. Abaqus `.inp` with EVERY grit

This is the deliverable. Each distinct grain shape is written once as a `*Part`; every grit
on the wheel is an `*Instance` of it with its own translation and rotation. That is what
lets a 100,000-grit model fit in ~26 MB instead of gigabytes.

`--verify` re-reads the finished file from disk and checks every grit was reconstructed at
the right radius, angle and protrusion.


In [ ]:
cmd = ('python -W ignore -m semgrit wheel results -o abaqus'
       ' --name wheel_abaqus' + WHEEL_ARGS +
       ' --radial-divisions 4 --axial-divisions 10 --circ-divisions-per-deg 2 --verify')
print(cmd, chr(10))
!{cmd}


### Independent audit: is every grit really in the `.inp`?

Parses the finished file with a separate reader -- no pickles, no in-memory model -- rebuilds
each grit in assembly coordinates, and checks it straddles the wheel surface.


In [ ]:
import re, math, numpy as np, glob

INP = sorted(glob.glob('abaqus/*.inp'))[0]
txt = open(INP, encoding='ascii', errors='replace').read()

parts = {}
for m in re.finditer(r'^\*Part,\s*name=([^\s,]+)(.*?)^\*End Part', txt, re.M | re.S):
    nm, body = m.group(1), m.group(2)
    nodes = {}
    q = re.search(r'^\*Node\s*$(.*?)(?=^\*)', body, re.M | re.S)
    if q:
        for ln in q.group(1).strip().split('\n'):
            v = [x.strip() for x in ln.split(',')]
            if len(v) >= 4:
                nodes[int(v[0])] = (float(v[1]), float(v[2]), float(v[3]))
    parts[nm] = nodes
grain_parts = {k: v for k, v in parts.items() if k.startswith('GRAIN-')}

inst = re.findall(r'\*Instance,\s*name=([^\s,]+),\s*part=([^\s,]+)\s*\n([^*]*?)\*End Instance', txt)
grits = []
for nm, pt, data in inst:
    if not nm.startswith('G-'):
        continue
    rows = [[float(x) for x in ln.split(',') if x.strip()]
            for ln in data.strip().split('\n') if ln.strip()]
    a = np.array(rows[1][0:3]) if len(rows) > 1 and len(rows[1]) >= 7 else None
    b = np.array(rows[1][3:6]) if a is not None else None
    ang = rows[1][6] if a is not None else 0.0
    grits.append((pt, np.array(rows[0][:3]), a, b, ang))

def rot(axis, deg):
    n = np.linalg.norm(axis)
    if n < 1e-15:
        return np.eye(3)
    x, y, z = axis / n; c, s = math.cos(math.radians(deg)), math.sin(math.radians(deg))
    return np.array([[c+x*x*(1-c), x*y*(1-c)-z*s, x*z*(1-c)+y*s],
                     [y*x*(1-c)+z*s, c+y*y*(1-c), y*z*(1-c)-x*s],
                     [z*x*(1-c)-y*s, z*y*(1-c)+x*s, c+z*z*(1-c)]])

cache = {k: np.array([v[i] for i in sorted(v)]) for k, v in grain_parts.items()}
R = wheel_diameter_mm / 2.0
rmax = np.empty(len(grits)); rmin = np.empty(len(grits))
th = np.empty(len(grits)); zc = np.empty(len(grits))
for i, (pt, t, a, b, ang) in enumerate(grits):
    P = cache[pt] + t
    if a is not None and abs(ang) > 1e-12:
        P = (P - a) @ rot(b - a, ang).T + a
    rr = np.hypot(P[:, 0], P[:, 1]); rmax[i] = rr.max(); rmin[i] = rr.min()
    th[i] = math.degrees(math.atan2(P[:, 1].mean(), P[:, 0].mean())); zc[i] = P[:, 2].mean()

print(f'file                       : {INP}')
print(f'grain *Part definitions    : {len(grain_parts)}')
print(f'grit *Instance blocks      : {len(grits):,}')
print(f'every grit straddles the OD: {bool(((rmax > R) & (rmin < R)).all())}')
print(f'protrusion                 : {(rmax-R).min()*1000:.3f} .. {(rmax-R).max()*1000:.3f} um')
print(f'theta                      : {th.min():.4f} .. {th.max():.4f} deg '
      f'(inside 0-{sector_deg}: {bool(((th>=-1e-6)&(th<=sector_deg+1e-6)).all())})')
print(f'axial z                    : {zc.min():.4f} .. {zc.max():.4f} mm')
h, _ = np.histogram(th, bins=6, range=(0, sector_deg))
print(f'grits per band across sector: {h.tolist()}  (even = not clumped)')


## 7. CAD of the wheel WITH grits on it

Set **`cad_grits = 0` to export every grit**, regardless of size.

Be aware of the cost before you do: a STEP B-rep runs about **430 bytes per planar face**,
and a CAD kernel is far slower per solid *body* than an FE solver is per element. Guide:

| grits | STEP size | opens in SOLIDWORKS? |
|---|---|---|
| 250 | ~16 MB | yes |
| 1,000 | ~62 MB | yes |
| 5,000 | ~300 MB | slow |
| 20,000 | ~1.4 GB | probably not |
| 100,000 | ~6.3 GB | no |

The writer streams to disk, so file size is limited by your disk rather than by RAM. The
`.inp` always contains every grit whatever you choose here.

Note the scale: your grits are microns on a 200 mm wheel, roughly **1:40,000**. Zoomed out
to the whole sector they are far below one pixel; **zoom onto the outer diameter** to see
them.


In [ ]:
cad_grits = 1000  #@param {type:"integer"}
also_write_stl = True  #@param {type:"boolean"}

cmd = ('python -W ignore -m semgrit wheel results -o cad'
       ' --name wheel_cad' + WHEEL_ARGS +
       f' --step --step-max-grains {cad_grits}'
       + (' --stl --stl-max-grains 20000' if also_write_stl else '')
       + ' --circ-divisions-per-deg 6 --verify')
print(cmd, chr(10))
!{cmd}


### Extra CAD: individual grits, and a dense surface patch

At the whole-wheel scale grits are invisible, so these two are the useful CAD views:
single grits to inspect shape, and a small patch at **true** concentration to see what the
wheel surface really looks like.


In [ ]:
import os, pickle, numpy as np
from semgrit.wheel import WheelSpec, GrainPopulationSpec, build_wheel
from semgrit.step import (StepWriter, StepExportOptions, write_wheel_step,
                          write_grains_step, check_step_solids)
os.makedirs('cad', exist_ok=True)
lib = pickle.load(open('results/grain_library.pkl', 'rb'))
sol = lib['solids']

big = max(sol, key=lambda s: s.mesh_volume_um3)
w = StepWriter('cad/single_grain.step', 'single_grain')
w.add_faceted_solid(big.vertices - big.centroid_um, big.faces, f'GRAIN_{big.grain_id}')
i = w.finalize()
print(f'cad/single_grain.step         {i["size_bytes"]/1024:8.1f} KB  1 body, {len(big.faces)} faces')

i = write_grains_step('cad/all_measured_grains.step', sol, max_grains=0)
print(f'cad/all_measured_grains.step  {i["size_bytes"]/1e6:8.2f} MB  {i["n_grain_bodies"]} bodies (microns)')

spec = WheelSpec(diameter_mm=wheel_diameter_mm, width_mm=0.08, sector_deg=0.05,
                 rim_depth_mm=0.02)
m = build_wheel(spec, sol, GrainPopulationSpec(concentration=100.0, seed=20260728))
i = write_wheel_step('cad/surface_patch_true_density.step', m,
                     StepExportOptions(max_grains=0, name='surface_patch'))
print(f'cad/surface_patch_true_density.step {i["size_bytes"]/1e6:6.2f} MB  '
      f'{m.achieved_grains} grits at {m.stats["achieved_areal_density_per_mm2"]:.0f}/mm2')
for f in ('cad/single_grain.step', 'cad/all_measured_grains.step',
          'cad/surface_patch_true_density.step'):
    a = check_step_solids(f)
    print(f'   audit {os.path.basename(f):36s} {"PASS" if a["ok"] else "FAIL"}  '
          f'{a["n_solids"]} solids, {a["n_faces"]} faces')


## 8. Download everything

Small files download directly. Anything over a few hundred MB is impractical through the
browser -- mount Google Drive and copy it there instead (the cell handles both).


In [ ]:
import os, glob, shutil
from google.colab import files

targets = sorted(glob.glob('abaqus/*') + glob.glob('cad/*'))
print('generated files:')
for f in targets:
    print(f'   {os.path.getsize(f)/1e6:9.2f} MB  {f}')

BIG = 300e6
big = [f for f in targets if os.path.getsize(f) > BIG]
small = [f for f in targets if os.path.getsize(f) <= BIG]

shutil.make_archive('measurements', 'zip', 'results')
for f in ['measurements.zip'] + small:
    files.download(f)

if big:
    print('\nthese are too large for a browser download:')
    for f in big:
        print(f'   {os.path.getsize(f)/1e9:.2f} GB  {f}')
    print('\nmount Drive and copy them instead:')
    print('   from google.colab import drive; drive.mount("/content/drive")')
    print('   !cp -v ' + ' '.join(big) + ' /content/drive/MyDrive/')


## Optional: full verification suite

36 checks: analytic ground-truth unit tests (circle/ellipse/square/L-shape measurements,
prism-to-tet partition by Monte Carlo, mesh conformity, edge-blunting against the exact
(4-pi)r^2 corner area, STEP volume round-trip, hex volumes vs analytic), integration across
your images, wheel assembly at five sector/element combinations, STEP and STL exports
checked against the FE model, plus Abaqus round-trips and determinism.

Two are deliberate negative controls, so the suite cannot pass vacuously: an inward-wound
STEP solid must be *rejected*, and a saturated packing must reject colliding grains.


In [ ]:
!python -W ignore verify_all.py "images/*.tif"
